In [0]:
# =============================================================================
# NOTEBOOK: 02d_silver_trending_coins.py
# LAYER:    Silver
# PURPOSE:  Flatten the deeply nested trending API response into a clean
#           Silver table with one row per trending coin per day.
#
# SOURCE:   bronze/trending_raw      (Delta table, path-based)
# OUTPUT:   silver/trending_coins    (Delta table, path-based, MERGE upsert)
#
# TRENDING PAYLOAD STRUCTURE (from Bronze):
#   payload : {
#     "coins": [
#       { "item": { "id": "pepe", "name": "Pepe", "symbol": "PEPE",
#                   "market_cap_rank": 42, "score": 0, "thumb": "https://..." } },
#       { "item": { ... } },
#       ...   (up to 7 items)
#     ]
#   }
#
# NESTING CHALLENGE:
#   - payload is a STRUCT (Autoloader inferred)
#   - payload.coins is an ARRAY of STRUCT
#   - each element has an "item" STRUCT nested inside it
#   We must explode coins, then access .item to reach the actual fields.
#
# WHAT THIS NOTEBOOK DOES:
#   1. Read latest Bronze batch
#   2. Extract trend_run_date from ingestion_timestamp
#   3. Explode payload.coins → one row per {item: {...}} struct
#   4. Extract item fields (id, name, symbol, score, market_cap_rank)
#   5. Cast types, add silver_processed_timestamp
#   6. Quality filters (no null coin_id or trend_run_date)
#   7. Dedup on trend_run_date + coin_id
#   8. MERGE into silver/trending_coins
#   9. OPTIMIZE + Z-ORDER
#
# SILVER TABLE: silver_trending_coins
#   Grain: one row per trending position per day (max 7 rows per day)
#   Primary key: trend_run_date + coin_id

In [0]:
%run ../connection

In [0]:
%run ./config

In [0]:
%run ./silver_utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DateType
 
adls_name = "adlsnewhp"
init_silver_config(adls_name)
 
logger = get_logger("silver_trending_coins")
 
logger.info("=" * 70)
logger.info("Silver: trending_coins — START")
logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
logger.info(f"  Bronze in : {BronzePaths.TRENDING}")
logger.info(f"  Silver out: {SilverPaths.TRENDING_COINS}")
logger.info("=" * 70)
 

In [0]:
# =============================================================================
# CELL 2 — READ BRONZE (latest batch)
# =============================================================================
 
bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.COINS_MARKETS, WatermarkPaths.COINS_MARKETS,
    WatermarkPaths.WATERMARK_TABLE, logger
)
 
assert_required_columns(
    bronze_df,
    required_cols=["payload", "ingestion_timestamp", "pipeline_run_id"],
    table_name="trending_raw",
    logger=logger
)

In [0]:
bronze_df.display()

In [0]:
# =============================================================================
# CELL 3 — EXPLODE THE NESTED COINS ARRAY
#
# Step 1: Select ingestion_timestamp (used to derive trend_run_date)
#         and payload.coins (the array of {item: {...}} structs)
# Step 2: explode payload.coins → one row per coin entry
#         Each exploded row = { ingestion_timestamp, entry: {item: {...}} }
# Step 3: Access entry.item to reach the actual coin fields
#
# WHY USE ingestion_timestamp FOR trend_run_date?
#   The API does not return a date field in the trending response —
#   it's always "current" data. We derive the date from when our pipeline
#   collected it (ingestion_timestamp from the Bronze envelope).
#   This correctly records "trending on date X" for historical analysis.
# =============================================================================
 
logger.info("CELL 3: Exploding payload.coins → one row per trending coin")
 
exploded_df = (
    bronze_df
    .select(
        F.to_date(F.col("ingestion_timestamp")).alias("trend_run_date"),
        F.col("pipeline_run_id").cast(StringType()),
        # payload.coins is an ARRAY of STRUCT {item: {id, name, ...}}
        F.explode(F.col("payload.coins")).alias("entry")
    )
)
 
logger.info(f"  Rows after explode (should be ~7): {exploded_df.count():,}")
 
 

In [0]:
exploded_df.display()

In [0]:
# =============================================================================
# CELL 4 — EXTRACT ITEM FIELDS AND CAST TYPES
#
# entry.item.field_name accesses nested struct fields.
# score = position in the trending list (0 = #1 trending).
# We add 1 to score to make trend_position 1-indexed (human-readable).
# =============================================================================
 
logger.info("CELL 4: Extracting item fields and casting types")
 
typed_df = (
    exploded_df
    .select(
        F.col("trend_run_date").cast(DateType()),             # Primary key part 1
 
        # score is 0-based. We add 1 so position 1 = hottest trending.
        (F.col("entry.item.score").cast(IntegerType()) + 1)
         .alias("trend_position"),
 
        F.col("entry.item.id")
         .cast(StringType())
         .alias("coin_id"),                                    # Primary key part 2
 
        F.col("entry.item.name")
         .cast(StringType())
         .alias("coin_name"),
 
        F.upper(F.col("entry.item.symbol"))
         .cast(StringType())
         .alias("coin_symbol"),
 
        # market_cap_rank can be null for very new or unlisted coins
        F.col("entry.item.market_cap_rank")
         .cast(IntegerType())
         .alias("market_cap_rank"),
 
        F.col("pipeline_run_id"),
    )
    .withColumn("silver_processed_timestamp", get_silver_timestamp(SilverConfig.RUN_TS))
)
 
raw_count = typed_df.count()
logger.info(f"  Rows after extraction: {raw_count:,}")
 
 

In [0]:
typed_df.display()

In [0]:
# =============================================================================
# CELL 5 — DATA QUALITY FILTERS
# Trending is a short list (max 7) — if coin_id or trend_run_date is null,
# we cannot identify the coin or place it on the timeline. Drop those rows.
# =============================================================================
 
logger.info("CELL 5: Applying data quality filters")
 
clean_df = (
    typed_df
    .filter(F.col("coin_id").isNotNull())
    .filter(F.col("trend_run_date").isNotNull())
    .filter(F.col("coin_name").isNotNull())
    # trend_position must be a valid positive number
    .filter(
        F.col("trend_position").isNotNull() &
        (F.col("trend_position") >= 1)
    )
)
 
clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")
 
validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "trending_coins",
    logger       = logger,
)

In [0]:
clean_df.display()

In [0]:
# =============================================================================
# CELL 6 — DEDUPLICATE
# =============================================================================
 
logger.info("CELL 6: Deduplicating on trend_run_date + coin_id")
 
dedup_df    = clean_df.dropDuplicates(MergeKeys.TRENDING_COINS)
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,})")

In [0]:
# =============================================================================
# CELL 7 — FINAL COLUMN REORDER
# =============================================================================
 
final_df = dedup_df.select(*SilverColumns.TRENDING_COINS)
log_schema(final_df, "trending_coins_final", logger)

In [0]:

# =============================================================================
# CELL 8 — DELTA MERGE
# =============================================================================
 
logger.info("CELL 8: MERGE into silver/trending_coins")
 
merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.TRENDING_COINS,
    merge_keys = MergeKeys.TRENDING_COINS,
    logger     = logger,
)

# Update watermark ONLY after successful MERGE.
# If MERGE failed, an exception would have already propagated — we never reach here.
update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.COINS_MARKETS,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 

In [0]:
 
# =============================================================================
# CELL 9 — OPTIMIZE + Z-ORDER
# Z-ORDER BY trend_run_date: Gold's trending enrichment joins by date.
# =============================================================================
 
logger.info("CELL 9: OPTIMIZE silver/trending_coins")
 
spark.sql(f"OPTIMIZE delta.`{SilverPaths.TRENDING_COINS}` ZORDER BY (trend_run_date)")
logger.info("  ✓ OPTIMIZE complete")

In [0]:
# =============================================================================
# CELL 10 — RUN LOG + COMPLETION
# =============================================================================
 
summary = {
    "notebook"                 : "02d_silver_trending_coins",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.TRENDING,
    "silver_target"            : SilverPaths.TRENDING_COINS,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}
 
write_run_log(summary, LogPaths.TRENDING_COINS, logger)
 
logger.info("=" * 70)
logger.info("Silver: trending_coins — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)